In [1]:
%load_ext autotime

import pandas as pd
import numpy as np

# Load data

In [2]:
df = pd.read_csv("outputs/1.0-discovery_results.csv")
df

,snapshot_id,snapshot_type,source,model,metadata_field,observed_value,description,source_level,discovery_value,reasoning
0,027_Jordan-Emergency-Food-Security-Project_fig...,figure,refugee,gpt-5.5,figure_title,Figure 2: Share of food expenditure in total e...,The title or caption identifying the figure an...,snapshot,high,The figure title provides the most direct sear...
1,027_Jordan-Emergency-Food-Security-Project_fig...,figure,refugee,gpt-5.5,indicator_name,Share of food expenditure in total expenditure...,The main measure or indicator represented in t...,snapshot,high,Indicator-level metadata enables retrieval of ...
2,027_Jordan-Emergency-Food-Security-Project_fig...,figure,refugee,gpt-5.5,subject_domain,Food expenditure; food security,The thematic domain or topical area represente...,both,high,The figure title and document metadata both co...
3,027_Jordan-Emergency-Food-Security-Project_fig...,figure,refugee,gpt-5.5,geographic_scope,"Jordan, inferred from parent document title an...","The country, region, or geographic area to whi...",document,high,Geographic context is essential for discovery ...
4,027_Jordan-Emergency-Food-Security-Project_fig...,figure,refugee,gpt-5.5,time_period,Not identifiable from this snapshot,"The date, year, or period covered by the data ...",snapshot,high,A data time period would be highly useful for ...
...,...,...,...,...,...,...,...,...,...,...
3036,document_10395051_table_002.png,table,prwp,gpt-5.5,significance_notation,"*, ** and *** denote significance at the 10 pe...",Notation used to indicate statistical signific...,snapshot,medium,Significance notation is needed to interpret c...
3037,document_10395051_table_002.png,table,prwp,gpt-5.5,robustness_adjustment,Heteroskedasticity-robust t(z)-statistics are ...,Adjustments or corrections applied to reported...,snapshot,medium,Inference adjustments improve reuse by indicat...
3038,document_10395051_table_002.png,table,prwp,gpt-5.5,endogenous_variable,Each of four informality measures,Variables treated as endogenous in instrumenta...,snapshot,medium,Endogeneity treatment is important for underst...
3039,document_10395051_table_002.png,table,prwp,gpt-5.5,instrumental_variables,Law and order; Business regulatory freedom; Av...,Instruments used in instrumental-variable regr...,snapshot,high,Instrumental variables are key metadata for di...


# Initial field profiles

In [3]:
from collections import Counter
import pyperclip


def df_to_clipboard(df, index=True):
    pyperclip.copy(df.to_markdown(index=index))


def top_n_values(lst, n, filter_keyword=None):
    if filter_keyword:
        lst = [x for x in lst if filter_keyword not in str(x).lower()]

    return [x for x, _ in Counter(lst).most_common(n)]

In [4]:
snapshot_count = df["snapshot_id"].nunique()

profiles = (
    df.groupby("metadata_field")
    .agg(
        count=("metadata_field", "count"),
        n_snapshots=("snapshot_id", "nunique"),
        support=("snapshot_id", lambda x: x.nunique() / snapshot_count),
        corpora_count=("source", "nunique"),
        figure_count=("snapshot_type", lambda x: (x == "figure").sum()),
        table_count=("snapshot_type", lambda x: (x == "table").sum()),
        source_snapshot_count=("source_level", lambda x: (x == "snapshot").sum()),
        source_document_count=("source_level", lambda x: (x == "document").sum()),
        source_both_count=("source_level", lambda x: (x == "both").sum()),
        top_observed_values=(
            "observed_value",
            lambda x: top_n_values(x, 5, filter_keyword="not identifiable"),
        ),
        n_unique_observed_values=(
            "observed_value",
            lambda x: x[
                ~x.str.contains("not identifiable", case=False, na=False)
            ].nunique(),
        ),
        top_description_values=("description", lambda x: top_n_values(x, 3)),
        top_reasoning_values=("reasoning", lambda x: top_n_values(x, 3)),
    )
    .sort_values("count", ascending=False)
)

profiles

,count,n_snapshots,support,corpora_count,figure_count,table_count,source_snapshot_count,source_document_count,source_both_count,top_observed_values,n_unique_observed_values,top_description_values,top_reasoning_values
metadata_field,,,,,,,,,,,,,
geographic_scope,192,192,0.914286,3,96,96,65,56,71,"[Lebanon, Djibouti, Uganda, Croatia, World / T...",121,[Country or geographic area to which the table...,[Geographic scope is essential for country-lev...
unit_of_measure,180,180,0.857143,3,98,82,180,0,0,"[Percent (%), Percent, US$, Millions, US$, Kil...",129,[Measurement unit used for the plotted values....,[Units are essential for interpreting and reus...
indicator_name,152,152,0.723810,3,89,63,152,0,0,"[Adjustment functions, Forcibly displaced and ...",140,[Name of the measured concept or indicator rep...,[Indicator names are central for searching and...
time_period,149,149,0.709524,3,86,63,142,4,3,"[2023, décembre 2023, Septembre 2023, 1994-200...",109,[Time period covered by the data in the snapsh...,[Time period enables temporal filtering and co...
row_dimension,102,102,0.485714,3,16,86,102,0,0,"[Territoires, Region, Expenditure Category, Co...",96,[The conceptual dimension represented by the t...,[Row dimension clarifies how the table is orga...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
cost_breakdown_type,1,1,0.004762,1,0,1,1,0,0,[Local and foreign cost breakdown; baseline co...,1,[Type of financial decomposition used to organ...,[Breakdown type helps users distinguish tables...
measure_labels,1,1,0.004762,1,0,1,1,0,0,[Credit Amount; Grant Amount; SML Amount; Guar...,1,[Named measures or value columns reported in t...,[Measure labels support precise discovery of t...
model_outcome_variables,1,1,0.004762,1,0,1,1,0,0,[Per capita expenditure; Per capita income],1,[The dependent or outcome variables reported i...,[Outcome variables are central for searching a...


In [5]:
# df_to_clipboard(profiles.head(10))

In [6]:
profiles.to_csv("outputs/3.0-ontology_input.csv")